# Merge and Upload

- 목적: 학습된 LoRA Adapter를 base model에 병합 후 Hugging Face model repo에 업로드
- 업로드 대상: merged model + tokenizer
- 제외 대상: dataset
- repo 공개 범위: 기본 private
- vLLM 배포용으로 adapter-only가 아니라 merged model 사용

In [ ]:
from pathlib import Path

import torch
from huggingface_hub import HfApi, login, whoami
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

In [ ]:
# 경로 및 repo 설정
adapter_path = Path("gemma4-e4b-civil-complaint-sft")
base_model_id = "google/gemma-4-E4B-it"
merged_path = Path("gemma4-e4b-civil-complaint-merged")
model_repo_name = "gemma4-e4b-civil-complaint-merged"
private_repo = True

if not adapter_path.exists():
    raise FileNotFoundError(f"LoRA Adapter 경로가 없습니다: {adapter_path}")

In [ ]:
# Hugging Face 로그인 및 repo ID 구성
login()

api = HfApi()
hf_username = whoami()["name"]
hf_repo_id = f"{hf_username}/{model_repo_name}"

print("HF repo:", hf_repo_id)
print("private:", private_repo)

In [ ]:
from pathlib import Path

import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, AutoProcessor
from huggingface_hub import HfApi, list_repo_files

merged_path = Path(merged_path)

fine_tuned_model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_path,
    device_map="auto",
    dtype=torch.bfloat16,
)

merged_model = fine_tuned_model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
processor = AutoProcessor.from_pretrained(base_model_id)

merged_model.save_pretrained(merged_path, safe_serialization=True)
tokenizer.save_pretrained(merged_path)
processor.save_pretrained(merged_path)

print("merged model saved:", merged_path)

local_files = sorted(p.name for p in merged_path.iterdir())
print("local files:")
print("\n".join(local_files))

required_files = [
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "processor_config.json",
]

missing_local_files = [name for name in required_files if name not in local_files]
if missing_local_files:
    raise RuntimeError(f"Missing local files: {missing_local_files}")

api = HfApi()

api.create_repo(
    repo_id=hf_repo_id,
    repo_type="model",
    private=private_repo,
    exist_ok=True,
)

api.upload_folder(
    repo_id=hf_repo_id,
    repo_type="model",
    folder_path=str(merged_path),
    commit_message="Upload merged Gemma4 model with processor config",
)

remote_files = list_repo_files(hf_repo_id)
print("remote files:")
print("\n".join(remote_files))

missing_remote_files = [name for name in required_files if name not in remote_files]
if missing_remote_files:
    raise RuntimeError(f"Missing remote files: {missing_remote_files}")

print("uploaded:", f"https://huggingface.co/{hf_repo_id}")